# 03. Cell-Type Reference: GEO Download, Segmentation, and Marker Discovery

Builds the cell-type methylation reference required for LLR digital sorting (notebook 04).

**Pipeline:**
1. Download ENCODE/GEO reference WGBS beta files (T-Cell, NK, Monocyte, Granulocyte, etc.)
2. Run `wgbstools segment` on the reference betas to identify differentially methylated blocks
3. Run `wgbstools find_markers` to select high-specificity CpG markers per cell type
4. Upload marker BED files to GCS (`results/markers/markers_S2/`)

**Outputs:**
- `markers_S2/coarse/Myeloid_vs_Lymphoid/Markers.Myeloid.bed` (and `.Lymphoid.bed`)
- `markers_S2/hier/{subtype}_vs_rest/Markers.{subtype}.bed` for T_Cell, B_Cell, NK_Cell, Monocyte, Granulocyte

---

## 1. Environment Setup

Verify wgbstools installation and NCBI/SRA access for reference data download.

In [ ]:
import os
import sys
import pandas as pd
import subprocess
import requests
import re
import urllib.parse

# Get strict variable names for AoU environment
project_id = os.environ['GOOGLE_PROJECT']
workspace_bucket = os.environ['WORKSPACE_BUCKET']

In [ ]:
%%bash

# 1. Create the .ncbi folder if it doesn't exist
mkdir -p /home/jupyter/.ncbi

# 2. Generate a UUID using Python and write the config file
# This config disables the "user interface" prompts that hang the notebook
# and sets a unique ID for the toolkit.
PYTHON_UUID=$(python3 -c 'import uuid; print(uuid.uuid4())')

printf '/LIBS/GUID = "%s"\n/config/default = "false"\n' "$PYTHON_UUID" > /home/jupyter/.ncbi/user-settings.mkfg

# 3. Verification: Check if the file was created
cat /home/jupyter/.ncbi/user-settings.mkfg

## 2. Reference Sample List

Build the list of GEO/SRA accessions for the blood cell-type WGBS reference panel.

In [ ]:
# GSM IDs
ids = []

# 1. T-Cells & NK Cells (GSM5652277 to GSM5652301)
ids.extend([f"GSM{i}" for i in range(5652277, 5652302)])

# 2. Monocytes (GSM5652302 to GSM5652304)
ids.extend([f"GSM{i}" for i in range(5652302, 5652305)])

# 3. Granulocytes (GSM5652313 to GSM5652315)
ids.extend([f"GSM{i}" for i in range(5652313, 5652316)])

# 4. B-Cells (GSM5652316 to GSM5652320)
ids.extend([f"GSM{i}" for i in range(5652316, 5652321)])

# Write to file
filename = "SRR_Acc_List.txt"
with open(filename, 'w') as f:
    f.write('\n'.join(ids))

print(f"Created '{filename}' with {len(ids)} samples.")
print(f"First 3: {ids[:3]}")
print(f"Last 3:  {ids[-3:]}")

In [ ]:
%%bash
echo "Current Directory:"
pwd
echo "Files in this directory:"
ls -lh SRR_Acc_List.txt 2>/dev/null || echo "--> FILE MISSING! Please upload 'SRR_Acc_List.txt' to this folder."

## 3. Download Reference Beta Files

Download pre-processed WGBS beta files from GEO using SRA toolkit. Each beta file encodes (methylated_count, total_count) pairs for all 29,152,891 hg38 CpGs.

In [ ]:
# --- CONFIGURATION ---
srr_list_file = "SRR_Acc_List.txt" 
bucket = os.getenv('WORKSPACE_BUCKET')
output_folder = f"{bucket}/data/processed_betas"

# Create output folder
os.system(f"gsutil -m mkdir -p {output_folder}")
# ---------------------

def get_https_download_link(gsm_id):
    """
    Visits the GEO page, finds the filename ending in .hg38.beta,
    and constructs the HTTPS download URL.
    """
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gsm_id}"
    try:
        r = requests.get(url)
        if r.status_code != 200: 
            print(f"  [!] HTTP {r.status_code} fetching page.")
            return None, None
        
        # Regex: Look for the filename in the HTML content
        # Pattern: GSM followed by numbers, then characters, ending in .hg38.beta
        # We explicitly look for 'hg38.beta' to ensure we get the right genome build
        pattern = r"(GSM\d+_[a-zA-Z0-9_\-]+\.hg38\.beta)"
        
        match = re.search(pattern, r.text)
        
        if match:
            filename = match.group(1)
            # Construct the official download URL
            # We must URL-encode the filename (e.g., spaces/underscores become %5F)
            encoded_name = urllib.parse.quote(filename)
            download_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?acc={gsm_id}&format=file&file={encoded_name}"
            return download_url, filename
            
        return None, None

    except Exception as e:
        print(f"  [!] Error: {e}")
        return None, None

# --- MAIN LOOP ---
if not os.path.exists(srr_list_file):
    print(" Error: ID list not found.")
else:
    with open(srr_list_file, 'r') as f:
        gsm_ids = [line.strip() for line in f if line.strip()]

    print(f"Found {len(gsm_ids)} samples. Starting HTTPS scrape...")

    for i, gsm in enumerate(gsm_ids):
        print(f"\n[{i+1}/{len(gsm_ids)}] Processing {gsm}...")
        
        # 1. Get the link
        link, filename = get_https_download_link(gsm)
        
        if not link:
            print(f"   Could not find .hg38.beta filename on page. Skipping.")
            continue
            
        print(f"  > Found: {filename}")
        
        # 2. Check if already exists in bucket
        check = os.system(f"gsutil ls {output_folder}/{filename} > /dev/null 2>&1")
        if check == 0:
            print("  > File already in bucket. Skipping.")
            continue
            
        # 3. Stream Download
        print(f"  > Downloading...")
        # curl -L follows redirects, -o - outputs to stdout, piped to gsutil
        cmd = f"curl -s -L '{link}' | gsutil cp - {output_folder}/{filename}"
        
        if os.system(cmd) == 0:
            print(" Success.")
        else:
            print(" Download failed.")

    print("\nDone! All files should be in your bucket.")

In [ ]:
import os
import subprocess

# --- CONFIGURATION ---
bucket = os.getenv('WORKSPACE_BUCKET')
beta_folder = f"{bucket}/data/processed_betas"
# ---------------------

print(f"Checking {beta_folder}...")

# 1. Get List of Files with Sizes
# gsutil du -h prints human-readable sizes
try:
    result = subprocess.check_output(f"gsutil du -h {beta_folder}/*.beta", shell=True).decode('utf-8')
    lines = result.strip().split('\n')
    
    valid_files = [l for l in lines if l.strip()]
    count = len(valid_files)
    
    print(f"\n Found {count} beta files.")
    
    if count == 0:
        print(" No files found! Did the download finish?")
    else:
        # 2. Check Sizes (Sample the first 5)
        print("\n--- Sample File Sizes (Should be ~50-80 MB) ---")
        for line in valid_files[:5]:
            print(line)
        
        if count == 36:
            print("\n PERFECT count! You have all 36 samples.")
        else:
            print(f"\n WARNING: You have {count} samples (expected 36).")
            
except subprocess.CalledProcessError:
    print(" Error reading bucket. Check permissions.")

In [ ]:
%%bash
# Show free space in your home folder (mounted at /home/jupyter)
df -h /home/jupyter

## 4. Load and Validate Reference Betas

Load each beta file and confirm the expected 29,152,891 CpG sites.

In [ ]:
import os
import sys
import pandas as pd
import subprocess
# Get strict variable names for AoU environment
project_id = os.environ['GOOGLE_PROJECT']
workspace_bucket = os.environ['WORKSPACE_BUCKET']

# Set paths for our tools
# (We will unpack them here in a moment)
HOME = os.getcwd()
TOOL_DIR = f"{HOME}/wgbs_tools"
TOOL_BIN = f"{TOOL_DIR}/wgbstools"
GENOME_FILE = f"{HOME}/hg38.fasta"

print(f" Environment Configured.")
print(f"   Bucket: {workspace_bucket}")
print(f"   Tools will be at: {TOOL_BIN}")



resources = {
    "wgbstools": f"{workspace_bucket}/resources/wgbstools_package.tar.gz",
    "hg38": f"{workspace_bucket}/resources/hg38_bundle.tar.gz"
}

print(" Checking Local Resources...")

# 1. Check/Install Tools
if not os.path.exists(TOOL_BIN):
    print("     Fetching wgbstools from bucket...")
    os.system(f"gsutil cp {resources['wgbstools']} .")
    os.system("tar -xzf wgbstools_package.tar.gz")
    
    # Fix permissions
    os.system(f"chmod +x {TOOL_BIN}")
else:
    print("    wgbstools found locally.")

# 2. Check/Install Genome
if not os.path.exists(GENOME_FILE):
    print("     Fetching hg38 genome from bucket...")
    os.system(f"gsutil cp {resources['hg38']} .")
    os.system("tar -xzf hg38_bundle.tar.gz")
else:
    print("    hg38.fasta found locally.")

# 3. Initialize Genome (Required for fresh notebook)
print("    Initializing Genome Config...")
os.system(f"{TOOL_BIN} init_genome hg38 --fasta_path {GENOME_FILE} -f")

print(" System Ready.")

In [ ]:
import os, glob

BETA_GCS  = f"{workspace_bucket}/data/processed_betas"
BETA_DIR  = "data/betas"
os.makedirs(BETA_DIR, exist_ok=True)

# Pull from bucket to local (idempotent)
os.system(f"gsutil -m cp '{BETA_GCS}/*.beta' '{BETA_DIR}/'")

print("Local beta files:", len(glob.glob(f"{BETA_DIR}/*.beta")))

In [ ]:
import os, glob

TARGET_SITES = 29152891
BETA_DIR = "data/betas"
files = sorted(glob.glob(f"{BETA_DIR}/*.beta"))

if not files:
    raise FileNotFoundError(f"No .beta files found in {BETA_DIR}. Did you gsutil cp from bucket?")

bytes_per_site = 2  # typical for wgbstools beta
keep_bytes = TARGET_SITES * bytes_per_site

pruned = 0
for fp in files:
    sz = os.path.getsize(fp)
    if sz < keep_bytes:
        raise ValueError(f"{os.path.basename(fp)} is smaller than expected ({sz} < {keep_bytes}). Wrong TARGET_SITES or format.")
    if sz > keep_bytes:
        with open(fp, "r+b") as f:
            f.truncate(keep_bytes)
        pruned += 1

print("Pruned files:", pruned, "out of", len(files))

In [ ]:
import glob, os

BETA_DIR = os.path.join(os.getcwd(), "data", "betas")
beta_paths = sorted(glob.glob(os.path.join(BETA_DIR, "*.beta")))

print("Found local betas:", len(beta_paths))
assert len(beta_paths) > 0, f"No .beta files in {BETA_DIR}"

with open("paths.txt", "w") as f:
    f.write("\n".join(beta_paths) + "\n")

print("Wrote paths.txt. Example:", beta_paths[0])
print("File exists:", os.path.exists(beta_paths[0]))

## 5. Segmentation: `wgbstools segment`

Identify differentially methylated blocks across all reference beta files. This produces `segmented_blocks.bed` used as the universe for `find_markers`.

In [ ]:
import os, subprocess

TOOL_BIN = f"{os.getcwd()}/wgbs_tools/wgbstools"
PATHS_FILE = "paths.txt"
OUTPUT_BED = "segmented_blocks.bed"

assert os.path.exists(PATHS_FILE), "paths.txt missing"

cmd = [
    TOOL_BIN, "segment",
    "--beta_file", PATHS_FILE,
    "--out_path", OUTPUT_BED,   # same as -o
    "--genome", "hg38",
    "--threads", "2"
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print(" segment done. First 3 lines:")
subprocess.run(["head", "-n", "3", OUTPUT_BED], check=True)

In [ ]:
import subprocess
WGBS = f"{os.getcwd()}/wgbs_tools/wgbstools"
subprocess.run([WGBS, "segment", "-h"], check=False)

In [ ]:
import os, glob, pandas as pd

BETA_DIR = os.path.join(os.getcwd(), "data", "betas")
betas = sorted(glob.glob(os.path.join(BETA_DIR, "*.beta")))
assert len(betas) == 36, f"Expected 36 betas, found {len(betas)}"

rows = []
for p in betas:
    fn = os.path.basename(p)
    sample = fn.rsplit(".beta", 1)[0]   # IMPORTANT: keep ".hg38" part

    if "Blood-Monocytes" in fn or "Blood-Granulocytes" in fn:
        grp = "Myeloid"
    else:
        grp = "Lymphoid"

    rows.append({"sample": sample, "group": grp})

labels = pd.DataFrame(rows)
labels.to_csv("labels_coarse.csv", index=False)

print(" wrote labels_coarse.csv")
print(labels["group"].value_counts())
print(labels.head())

In [ ]:
import os, glob, pandas as pd

BETA_DIR = os.path.join(os.getcwd(), "data", "betas")
betas = sorted(glob.glob(os.path.join(BETA_DIR, "*.beta")))
assert len(betas) == 36, f"Expected 36 betas, found {len(betas)}"

rows = []
for p in betas:
    fn = os.path.basename(p)
    sample = fn.rsplit(".beta", 1)[0]   # IMPORTANT: keep ".hg38"

    if "Blood-T-" in fn:
        group = "T"
    elif "Blood-NK" in fn:
        group = "NK"
    elif "Blood-B" in fn:
        group = "B"
    elif "Blood-Monocytes" in fn:
        group = "Monocytes"
    elif "Blood-Granulocytes" in fn:
        group = "Granulocytes"
    else:
        raise ValueError(f"Unrecognized type: {fn}")

    rows.append({"sample": sample, "group": group})

labels_fine = pd.DataFrame(rows)
labels_fine.to_csv("labels_fine.csv", index=False)
print(labels_fine["group"].value_counts())
labels_fine.head()

## 6. Coarse Markers: Myeloid vs Lymphoid

Run `wgbstools find_markers` twice:
- **Myeloid target**, Lymphoid background
- **Lymphoid target**, Myeloid background

Outputs go to `markers_coarse_S2_Myeloid_vs_Lymphoid/` and `markers_coarse_S2_Lymphoid_vs_Myeloid/`.

In [ ]:
import os, subprocess

TOOL_BIN = f"{os.getcwd()}/wgbs_tools/wgbstools"
OUT_DIR = "markers_coarse_S2_Myeloid_vs_Lymphoid"
os.system(f"rm -rf {OUT_DIR} && mkdir -p {OUT_DIR}")

cmd = [
    TOOL_BIN, "find_markers",
    "--blocks_path", "segmented_blocks.bed",
    "--groups_file", "labels_coarse.csv",
    "--beta_list_file", "paths.txt",
    "--out_dir", OUT_DIR,
    "--targets", "Myeloid",
    "--background", "Lymphoid",
    "--min_cov", "5",
    "--na_rate_tg", "0.334",
    "--na_rate_bg", "0.334",
    "--delta_means", "0.25",
    "--delta_quants", "0",
    "--tg_quant", "0.3",
    "--bg_quant", "0.03",
    "--pval", "0.05",
    "--chunk_size", "150000",
    "--threads", "8",
    "--header"
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=False)

print("\nOutputs:")
os.system(f"ls -lh {OUT_DIR} | head -n 50")

In [ ]:
import os, subprocess

TOOL_BIN = f"{os.getcwd()}/wgbs_tools/wgbstools"
OUT_DIR = "markers_coarse_S2_Lymphoid_vs_Myeloid"
os.system(f"rm -rf {OUT_DIR} && mkdir -p {OUT_DIR}")

cmd = [
    TOOL_BIN, "find_markers",
    "--blocks_path", "segmented_blocks.bed",
    "--groups_file", "labels_coarse.csv",
    "--beta_list_file", "paths.txt",
    "--out_dir", OUT_DIR,
    "--targets", "Lymphoid",
    "--background", "Myeloid",
    "--min_cov", "5",
    "--na_rate_tg", "0.334",
    "--na_rate_bg", "0.334",
    "--delta_means", "0.25",
    "--delta_quants", "0",
    "--tg_quant", "0.3",
    "--bg_quant", "0.03",
    "--pval", "0.05",
    "--chunk_size", "150000",
    "--threads", "8",
    "--header"
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=False)

print("\nOutputs:")
os.system(f"ls -lh {OUT_DIR} | head -n 50")

In [ ]:
%%bash
set -euo pipefail

echo "WORKSPACE_BUCKET=$WORKSPACE_BUCKET"

for DIR in markers_coarse_S2_Myeloid_vs_Lymphoid markers_coarse_S2_Lymphoid_vs_Myeloid
do
  DEST="$WORKSPACE_BUCKET/results/markers/$DIR"
  echo ""
  echo "Uploading $DIR -> $DEST"

  # rsync is nice because it creates the prefix implicitly and can be rerun safely
  gsutil -m rsync -r "$DIR" "$DEST"

  echo " Uploaded. Listing:"
  gsutil ls "$DEST/**" | head -n 50
done

In [ ]:
%%bash
ls -lh markers_coarse_S2_Myeloid_vs_Lymphoid/*.bed 2>/dev/null || echo "No BEDs in Myeloid_vs_Lymphoid"
ls -lh markers_coarse_S2_Lymphoid_vs_Myeloid/*.bed 2>/dev/null || echo "No BEDs in Lymphoid_vs_Myeloid"

In [ ]:
import os, glob, subprocess

OUT_DIR = "markers_coarse_S2_Myeloid_vs_Lymphoid"

# pick a reasonable default, else auto-detect
candidate = os.path.join(OUT_DIR, "Markers.Myeloid.bed")
if os.path.exists(candidate):
    OUT_FILE = candidate
else:
    beds = sorted(glob.glob(os.path.join(OUT_DIR, "Markers.*.bed")))
    if not beds:
        raise FileNotFoundError(f"No Markers.*.bed found in {OUT_DIR}. Contents:\n{os.listdir(OUT_DIR)}")
    OUT_FILE = beds[0]

print("Using:", OUT_FILE)
print("\nTop 5 (non-header) lines:\n")
subprocess.run(["bash","-lc", f"grep -v '^#' '{OUT_FILE}' | head -n 5"], check=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(OUT_FILE, sep="\t", comment="#", header=None)
print("Rows:", len(df), "Cols:", df.shape[1])

# columns based on your observed output
# 0 chrom, 1 start, 2 end, 3 startCpG, 4 endCpG, 5 target_group, 6 region_str,
# 7 nCpGs_str, 8 bp_str, 9 mean_target, 10 mean_background, 11 delta_means (already), then stats/pvals...
df["delta_calc"] = (df[9] - df[10]).abs()

# parse sizes like "8CpGs" and "275bp"
df["n_cpg"] = df[7].astype(str).str.replace("CpGs", "", regex=False).astype(int)
df["bp_len"] = df[8].astype(str).str.replace("bp", "", regex=False).astype(int)

# best markers
top = df.sort_values("delta_calc", ascending=False).head(10).copy()
top["Location"] = df[0].astype(str) + ":" + df[1].astype(int).astype(str) + "-" + df[2].astype(int).astype(str)

print(top[[0,1,2,7,8,9,10,"delta_calc"]].head(10))

# plot top 10 means
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(top))
w = 0.4
ax.bar(x, top[9], width=w, label="Myeloid")
ax.bar(x + w, top[10], width=w, label="Lymphoid")
ax.set_xticks(x + w/2)
ax.set_xticklabels(top["Location"], rotation=45, ha="right")
ax.set_ylabel("Methylation level")
ax.set_title("Top 10 marker blocks by |mean(Myeloid) - mean(Lymphoid)|")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Delta distribution
plt.figure(figsize=(8,5))
plt.hist(df["delta_calc"].values, bins=80)
plt.xlabel("|mean(Myeloid) - mean(Lymphoid)|")
plt.ylabel("Count of marker blocks")
plt.title("Delta distribution (marker blocks)")
plt.tight_layout()
plt.show()

# CpGs per block
plt.figure(figsize=(8,5))
plt.hist(df["n_cpg"].values, bins=80)
plt.xlabel("CpGs per block")
plt.ylabel("Count of marker blocks")
plt.title("Marker block CpG-count distribution")
plt.tight_layout()
plt.show()

# bp length per block (often heavy-tailed, log scale helps)
plt.figure(figsize=(8,5))
plt.hist(np.log10(df["bp_len"].values + 1), bins=80)
plt.xlabel("log10(bp_len + 1)")
plt.ylabel("Count of marker blocks")
plt.title("Marker block length distribution (log10 scale)")
plt.tight_layout()
plt.show()

## 7. Fine-Grained Hierarchy Markers

For each of the 5 subtypes (T_Cell, B_Cell, NK_Cell, Monocyte, Granulocyte), run one-vs-all `find_markers` within its coarse parent (Lymphoid or Myeloid). Outputs go to `markers_S2/hier/<subtype>_vs_rest/`.

In [ ]:
import os, glob

PATHS_FILE = "paths.txt"
BETA_DIR = "data/betas"

def write_paths():
    betas = sorted(glob.glob(os.path.join(BETA_DIR, "*.beta")))
    if not betas:
        raise FileNotFoundError(f"No .beta files found in {BETA_DIR}. You likely need to gsutil cp them from the bucket first.")
    with open(PATHS_FILE, "w") as f:
        for fp in betas:
            f.write(os.path.abspath(fp) + "\n")
    print(" wrote paths.txt with", len(betas), "entries")

# If missing OR empty, regenerate
if (not os.path.exists(PATHS_FILE)) or (os.path.getsize(PATHS_FILE) == 0):
    write_paths()

# Show first 3 lines (safe even if only 1-2 lines)
with open(PATHS_FILE) as f:
    for i in range(3):
        line = f.readline().strip()
        if not line:
            break
        print(line)

In [ ]:
import os
import pandas as pd

PATHS_FILE = "paths.txt"

with open(PATHS_FILE) as f:
    beta_paths = [line.strip() for line in f if line.strip()]
if not beta_paths:
    raise ValueError("paths.txt is empty. Recreate it from your local beta folder.")

def subtype_from_basename(bn: str) -> str:
    if "Blood-Granulocytes" in bn: return "Granulocyte"
    if "Blood-Monocytes" in bn:    return "Monocyte"
    if "Blood-NK" in bn:           return "NK_Cell"
    if "Blood-B" in bn:            return "B_Cell"
    if "Blood-T" in bn:            return "T_Cell"
    return "Unknown"

rows = []
for p in beta_paths:
    bn = os.path.basename(p)  # ...hg38.beta
    prefix = bn[:-5] if bn.endswith(".beta") else bn  # ...hg38   (strip only .beta)
    rows.append({"sample": prefix, "group": subtype_from_basename(bn)})

labels = "labels_subtypes_hg38prefix.csv"
df = pd.DataFrame(rows)
df.to_csv(labels, index=False)

print("Wrote:", labels)
print(df["group"].value_counts())
print("Example sample key:", df.iloc[0]["sample"])
print("Example beta basename:", os.path.basename(beta_paths[0]))

In [ ]:
import os, glob, subprocess, shutil, textwrap

TOOL_BIN   = f"{os.getcwd()}/wgbs_tools/wgbstools"
BED_FILE   = "segmented_blocks.bed"
PATHS_FILE = "paths.txt"

# S2 (LongReadAge) defaults
S2_ARGS = {
    "min_cov": "5",
    "na_rate_tg": "0.334",
    "na_rate_bg": "0.334",
    "delta_means": "0.25",
    "delta_quants": "0",
    "tg_quant": "0.3",
    "bg_quant": "0.03",
    "pval": "0.05",
    "chunk_size": "150000",
    "threads": "8",
}

def run_find_markers(out_dir, groups_file, targets, background):
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        TOOL_BIN, "find_markers",
        "--blocks_path", BED_FILE,
        "--groups_file", groups_file,
        "--beta_list_file", PATHS_FILE,
        "--out_dir", out_dir,
        "--targets", *targets,
        "--background", *background,
        "--min_cov", S2_ARGS["min_cov"],
        "--na_rate_tg", S2_ARGS["na_rate_tg"],
        "--na_rate_bg", S2_ARGS["na_rate_bg"],
        "--delta_means", S2_ARGS["delta_means"],
        "--delta_quants", S2_ARGS["delta_quants"],
        "--tg_quant", S2_ARGS["tg_quant"],
        "--bg_quant", S2_ARGS["bg_quant"],
        "--pval", S2_ARGS["pval"],
        "--chunk_size", S2_ARGS["chunk_size"],
        "--threads", S2_ARGS["threads"],
        "--header",
    ]

    log_path = os.path.join(out_dir, "run.log")
    p = subprocess.run(cmd, capture_output=True, text=True)

    with open(log_path, "w") as f:
        f.write("CMD:\n" + " ".join(cmd) + "\n\n")
        f.write("STDOUT:\n" + (p.stdout or "") + "\n\n")
        f.write("STDERR:\n" + (p.stderr or "") + "\n")

    beds = sorted(glob.glob(os.path.join(out_dir, "*.bed")))
    ok = (len(beds) > 0) and all(os.path.getsize(b) > 0 for b in beds)

    print(f"[{'OK' if ok else 'FAIL'}] {out_dir}  beds={len(beds)}  log={log_path}")
    if not ok:
        tail = (p.stderr or p.stdout).splitlines()[-25:]
        print("\n".join(tail))
    return ok, beds

In [ ]:
import os
import pandas as pd

with open(PATHS_FILE) as f:
    beta_paths = [line.strip() for line in f if line.strip()]
if not beta_paths:
    raise ValueError("paths.txt is empty")

def subtype_from_basename(bn: str) -> str:
    if "Blood-Granulocytes" in bn: return "Granulocyte"
    if "Blood-Monocytes" in bn:    return "Monocyte"
    if "Blood-NK" in bn:           return "NK_Cell"
    if "Blood-B" in bn:            return "B_Cell"
    if "Blood-T" in bn:            return "T_Cell"
    return "Unknown"

def coarse_from_subtype(st: str) -> str:
    if st in ["Monocyte", "Granulocyte"]: return "Myeloid"
    if st in ["T_Cell", "B_Cell", "NK_Cell"]: return "Lymphoid"
    return "Unknown"

rows = []
for p in beta_paths:
    bn = os.path.basename(p)                   # ...hg38.beta
    sample = bn[:-5] if bn.endswith(".beta") else bn  # ...hg38
    st = subtype_from_basename(bn)
    rows.append({"sample": sample, "subtype": st, "coarse": coarse_from_subtype(st)})

df = pd.DataFrame(rows)

labels_sub = "labels_subtypes_hg38prefix.csv"
labels_coarse = "labels_coarse_hg38prefix.csv"

df[["sample","subtype"]].rename(columns={"subtype":"group"}).to_csv(labels_sub, index=False)
df[["sample","coarse"]].rename(columns={"coarse":"group"}).to_csv(labels_coarse, index=False)

print("Wrote:", labels_sub, labels_coarse)
print("Subtype counts:\n", df["subtype"].value_counts())
print("Coarse counts:\n", df["coarse"].value_counts())
print("Example sample key:", df.iloc[0]["sample"])

## 8. Upload Marker BEDs to GCS and Validate

Rsync all marker directories to `results/markers/` in the workspace bucket.

In [ ]:
import shutil

BASE_OUT = "markers_S2"

# clean old outputs
shutil.rmtree(BASE_OUT, ignore_errors=True)
os.makedirs(BASE_OUT, exist_ok=True)

# 0) Coarse both directions
run_find_markers(
    out_dir=f"{BASE_OUT}/coarse_Myeloid_vs_Lymphoid",
    groups_file=labels_coarse,
    targets=["Myeloid"],
    background=["Lymphoid"],
)
run_find_markers(
    out_dir=f"{BASE_OUT}/coarse_Lymphoid_vs_Myeloid",
    groups_file=labels_coarse,
    targets=["Lymphoid"],
    background=["Myeloid"],
)

# group lists
subtypes = ["T_Cell", "B_Cell", "NK_Cell", "Monocyte", "Granulocyte"]

# 1) One-vs-all (each subtype vs all other subtypes)
for g in subtypes:
    bg = [x for x in subtypes if x != g]
    run_find_markers(
        out_dir=f"{BASE_OUT}/one_vs_all/{g}_vs_all",
        groups_file=labels_sub,
        targets=[g],
        background=bg,
    )

# 2) Hierarchical style contrasts (refinement layers)
hier = [
    # within myeloid
    ("Myeloid_Monocyte_vs_Granulocyte", ["Monocyte"], ["Granulocyte"]),
    ("Myeloid_Granulocyte_vs_Monocyte", ["Granulocyte"], ["Monocyte"]),
    ("Myeloid_Monocyte_vs_restMyeloid", ["Monocyte"], ["Granulocyte"]),
    ("Myeloid_Granulocyte_vs_restMyeloid", ["Granulocyte"], ["Monocyte"]),

    # within lymphoid
    ("Lymphoid_T_vs_restLymph", ["T_Cell"], ["B_Cell","NK_Cell"]),
    ("Lymphoid_B_vs_restLymph", ["B_Cell"], ["T_Cell","NK_Cell"]),
    ("Lymphoid_NK_vs_restLymph", ["NK_Cell"], ["T_Cell","B_Cell"]),
]

for name, tgt, bg in hier:
    run_find_markers(
        out_dir=f"{BASE_OUT}/hier/{name}",
        groups_file=labels_sub,
        targets=tgt,
        background=bg,
    )

In [ ]:
%%bash
set -euo pipefail
SRC="markers_S2"
DEST="$WORKSPACE_BUCKET/results/markers/$SRC"
gsutil -m rsync -r "$SRC" "$DEST"
echo " Saved to: $DEST"
gsutil ls "$DEST" | head